In [1]:
# Loads up the imports

from pathlib import Path
import random

import pandas as pd

import numpy as np
from PIL import Image
from skimage.feature import local_binary_pattern

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model


[TensorFlow DLL Diagnostic] Analyzing: C:\Users\rodri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "C:\Users\rodri\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: Uma rotina de inicialização da biblioteca de vínculo dinâmico (DLL) falhou.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
# Sets up the constants

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "training"
FEATURE_DIR = Path("outputs") / "features"
CLASSES = ["0", "1", "2", "3"]
VALIDATION_FRACTION = 0.20
SEED = 2100
LBP_POINTS = 8
LBP_RADIUS = 1
VGG16_IMAGE_SIZE = (224, 224)
VGG16_BATCH_SIZE = 16

In [ ]:
# Load images and create a DataFrame with their paths, classes, and patient IDs.

image_records = []

for class_label in CLASSES:
    class_folder = TRAIN_DIR / class_label
    image_paths = sorted(class_folder.glob("*.png"))

    for image_path in image_paths:
        image_records.append(
            {
                "path": image_path,
                "label": int(class_label),
                "patient": image_path.name.split("_", maxsplit=1)[0],
            }
        )

training_data = pd.DataFrame(image_records)

print(f"Imagens carregadas: {len(training_data)}")
print(f"Pacientes na base: {training_data['patient'].nunique()}")

Imagens carregadas: 593
Pacientes na base: 20


In [ ]:
# Split each class by patient so images from one patient cannot enter both sets.
# Select 20% of that class's patients for validation; with five patients, this selects one.
# A fixed seed makes the random patient selection reproducible.

random_generator = random.Random(SEED)
validation_patients = set()

for class_label, class_data in training_data.groupby("label", sort=True):
    class_patients = sorted(class_data["patient"].unique())
    validation_count = round(len(class_patients) * VALIDATION_FRACTION)
    selected_validation_patients = random_generator.sample(class_patients, validation_count)
    validation_patients.update(selected_validation_patients)
    print(
        f"Classe {class_label}: pacientes selecionados para validação: "
        f"{selected_validation_patients}"
    )

# Put all images from selected patients in validation; keep all remaining patients in training.

validation_mask = training_data["patient"].isin(validation_patients)
validation_data = training_data.loc[validation_mask].copy().reset_index(drop=True)
train_data = training_data.loc[~validation_mask].copy().reset_index(drop=True)

# Show how many images were assigned to each partition.

print(f"Pacientes de validação: {sorted(validation_patients)}")
print(f"Imagens para treino: {len(train_data)}")
print(f"Imagens para validação: {len(validation_data)}")

Classe 0: pacientes selecionados para validação: ['18']
Classe 1: pacientes selecionados para validação: ['24']
Classe 2: pacientes selecionados para validação: ['04']
Classe 3: pacientes selecionados para validação: ['19']
Pacientes de validação: ['04', '18', '19', '24']
Imagens para treino: 475
Imagens para validação: 118


In [ ]:
# Combine both partitions so they can be summarized in one table.

all_data = pd.concat(
    [
        train_data.assign(split="Treino"),
        validation_data.assign(split="Validação"),
    ],
    ignore_index=True,
)

# Count images and distinct patients for each partition and class.

split_summary = (
    all_data.groupby(["split", "label"])
    .agg(images=("path", "size"), patients=("patient", "nunique"))
)

# Display the counts to check the resulting split.

split_summary

images  patients
split     label                  
Treino    0         118         4
          1         117         4
          2         120         4
          3         120         4
Validação 0          28         1
          1          30         1
          2          30         1
          3          30         1

In [ ]:
# Compute LBP on grayscale images and use a normalized histogram as each feature vector.

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

lbp_feature_rows = []
histogram_bins = np.arange(0, LBP_POINTS + 3)

for image_path in training_data["path"]:
    grayscale_image = np.asarray(Image.open(image_path).convert("L"))
    lbp_image = local_binary_pattern(
        grayscale_image,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method="uniform",
    )
    histogram, _ = np.histogram(lbp_image.ravel(), bins=histogram_bins)
    lbp_feature_rows.append(histogram / histogram.sum())

lbp_feature_names = [f"lbp_{bin_index:02d}" for bin_index in range(LBP_POINTS + 2)]
lbp_features = pd.DataFrame(lbp_feature_rows, columns=lbp_feature_names)
lbp_csv = pd.concat(
    [training_data[["path", "label", "patient"]].reset_index(drop=True), lbp_features],
    axis=1,
)
lbp_csv["path"] = lbp_csv["path"].astype(str)

lbp_output_path = FEATURE_DIR / "lbp_features.csv"
lbp_csv.to_csv(lbp_output_path, index=False)

print(f"CSV LBP salvo em: {lbp_output_path.resolve()}")
print(f"Formato do CSV LBP: {lbp_csv.shape}")

In [ ]:
# Use ImageNet-pretrained VGG16 and extract the 4096 values from its penultimate (fc2) layer.
# The first run downloads the pretrained ImageNet weights.

vgg16_classifier = VGG16(weights="imagenet", include_top=True)
vgg16_feature_model = Model(
    inputs=vgg16_classifier.input,
    outputs=vgg16_classifier.layers[-2].output,
)

vgg16_feature_rows = []

for batch_start in range(0, len(training_data), VGG16_BATCH_SIZE):
    batch_data = training_data.iloc[batch_start : batch_start + VGG16_BATCH_SIZE]
    image_batch = np.stack(
        [
            np.asarray(
                Image.open(image_path)
                .convert("RGB")
                .resize(VGG16_IMAGE_SIZE, Image.Resampling.LANCZOS),
                dtype=np.float32,
            )
            for image_path in batch_data["path"]
        ]
    )
    image_batch = preprocess_input(image_batch)
    batch_features = vgg16_feature_model.predict(image_batch, verbose=0)
    vgg16_feature_rows.extend(batch_features)
    print(f"Imagens processadas pela VGG16: {len(vgg16_feature_rows)}/{len(training_data)}")

vgg16_feature_names = [f"vgg16_fc2_{feature_index:04d}" for feature_index in range(4096)]
vgg16_features = pd.DataFrame(vgg16_feature_rows, columns=vgg16_feature_names)
vgg16_csv = pd.concat(
    [training_data[["path", "label", "patient"]].reset_index(drop=True), vgg16_features],
    axis=1,
)
vgg16_csv["path"] = vgg16_csv["path"].astype(str)

vgg16_output_path = FEATURE_DIR / "vgg16_features.csv"
vgg16_csv.to_csv(vgg16_output_path, index=False)

print(f"CSV VGG16 salvo em: {vgg16_output_path.resolve()}")
print(f"Formato do CSV VGG16: {vgg16_csv.shape}")